# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahmir-028/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring (regression). I want a continuous prediction — how well a given piece of
content is likely to perform (its expected average SERP position) — not a fixed
set of classes (classification), and there's no need to group similar content
together (clustering). It's closer to ranking than classification, but I'm framing
it as scoring rather than pairwise ranking because I have an actual observed number
per item (avg SERP position), not just relative order — so I can train against that
value directly and get a usable output for any single new page, not just a ranked
list of a fixed batch.

In [1]:
task_type = "scoring (regression)"
print(f"Task type: {task_type}")

Task type: scoring (regression)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: avg_serp_position_30d — the average Google SERP position for the page's
primary target keyword, averaged over the last 30 days. This is a proxy, not the
true thing I ultimately care about (traffic/revenue from the page) — position is
an observed outcome (pulled from rank-tracking data), not a rule I defined myself,
but it's still a stand-in: two pages at the same position can drive very different
traffic depending on the keyword's search volume and SERP features.

In [2]:
import pandas as pd

df = pd.read_csv("/content/lane_content_placeholder.csv")
print("Target column:", "avg_serp_position_30d")
print(df["avg_serp_position_30d"].describe())

Target column: avg_serp_position_30d
count    150.000000
mean      18.992667
std       10.340531
min        1.000000
25%       11.725000
50%       18.050000
75%       26.700000
max       45.900000
Name: avg_serp_position_30d, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Mean Absolute Error (MAE), in position units, on a held-out test set. It's directly
interpretable — "the model is off by an average of N ranking positions" — which is
easy to defend to a non-technical stakeholder. Good is context-dependent: MAE well
below the natural spread of the target, and clearly better than a naive baseline
that just predicts the mean position for everyone.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X = df[["word_count", "keyword_density_pct", "backlinks", "readability_score", "page_load_time_sec"]]
y = df["avg_serp_position_30d"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression().fit(X_train, y_train)
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"Baseline MAE (positions): {mae:.2f}")

Baseline MAE (positions): 5.15


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one published content page (URL) in this lane, with its features and its
observed average ranking position.

In [4]:
print(f"Rows: {len(df)}, one row = one published content page (unit of analysis)")
df.head(3)

Rows: 150, one row = one published content page (unit of analysis)


,content_id,word_count,keyword_density_pct,backlinks,readability_score,page_load_time_sec,avg_serp_position_30d
0,C1000,1260,0.88,10,55.7,2.34,20.1
1,C1001,1694,0.61,15,73.1,2.54,3.2
2,C1002,1530,2.73,9,71.5,4.28,17.8


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single feature dominates, and the features interact and partially offset each
other — long word count generally helps, but not if load time is high; strong
backlinks can compensate for average readability, and so on. A fixed rule ("rank
well if word_count > 1500 AND backlinks > 10") would need a hand-tuned threshold
per feature and ignores these interactions, both of which drift as competition in
the lane changes over time. A model can learn the relative weight of each feature
from data instead of me guessing thresholds.

In [5]:
corr = df[["word_count","keyword_density_pct","backlinks","readability_score",
           "page_load_time_sec","avg_serp_position_30d"]].corr()["avg_serp_position_30d"]
print(corr)

word_count              -0.497740
keyword_density_pct     -0.192657
backlinks               -0.368254
readability_score       -0.291316
page_load_time_sec       0.297861
avg_serp_position_30d    1.000000
Name: avg_serp_position_30d, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.